# Create Unity Catalog HTTP connections for the Climb HLS MCP endpoints

Issues `CREATE CONNECTION ... TYPE HTTP` for each entry in the
`ENDPOINTS` dict below so the MCP endpoints are reachable from SQL as
`climb_hls_<app>_mcp` connections.

## How to use
1. Fill in the `ENDPOINTS` dict with the API Gateway URL for each
   deployed stack (leave entries empty / omit them if not deployed).
2. Set the `bearer_token` widget to the value printed by
   `scripts/qa-bearer-add-all.sh <customer>` from the repo root.
3. Run all cells.

Idempotent: re-running drops and recreates each connection, so it
doubles as a bearer-token rotation tool.

In [0]:
bearer_token = dbutils.secrets.get(scope="aichemy", key="climb_api")
connection_prefix = "climb_"

if not bearer_token:
    raise ValueError("bearer_token widget is required")
if not connection_prefix:
    raise ValueError("connection_prefix widget is required")

## Endpoints
Edit this dict to match what is deployed in the Climb AWS account.
Use the **origin only** (scheme + host); the `/mcp` path is added by
the connection's `base_path` option. Leave the value empty (or remove
the line) for apps that aren't deployed yet — they will be skipped.

In [0]:
ENDPOINTS: dict[str, str] = {
    "pubmed":         "https://pubmed.mcp.climb.ai",
    "chembl":         "https://chembl.mcp.climb.ai",
    "census":         "https://census.mcp.climb.ai",
    "biocontext":     "https://biocontext.mcp.climb.ai",
    "clinicaltrials": "https://clinicaltrials.mcp.climb.ai",
    "bioportal":      "https://bioportal.mcp.climb.ai",
    "opentargets":    "https://opentargets.mcp.climb.ai",
    "biorxiv":        "https://biorxiv.mcp.climb.ai",
    "openfda":        "https://openfda.mcp.climb.ai",
    "cms-coverage":   "https://cms-coverage.mcp.climb.ai",
}

## Create the connections
`CREATE CONNECTION` has no `OR REPLACE`, so we `DROP IF EXISTS` first
to make this idempotent. The token is supplied via the widget and
single-quote-escaped before being interpolated into SQL.

In [0]:
from urllib.parse import urlparse


def connection_name_for(app: str) -> str:
    # 'pubmed-mcp' -> '<prefix>pubmed_mcp'
    return connection_prefix + app.replace("-", "_")


def host_for(url: str) -> str:
    parsed = urlparse(url)
    if not parsed.scheme or not parsed.netloc:
        raise ValueError(f"endpoint must be like 'https://host'; got {url!r}")
    return f"{parsed.scheme}://{parsed.netloc}"


def create_connection(app: str, url: str) -> None:
    name = connection_name_for(app)
    host = host_for(url)
    safe_token = bearer_token.replace("'", "''")
    spark.sql(f"DROP CONNECTION IF EXISTS {name}")
    spark.sql(
        f"""
        CREATE CONNECTION {name} TYPE HTTP
        OPTIONS (
          host '{host}',
          port '443',
          base_path '/mcp',
          bearer_token '{safe_token}',
          is_mcp_connection 'true'
        )
        """
    )
    print(f"  ok  {name}  ->  {host}:443/mcp")


created = []
skipped = []
for app, url in ENDPOINTS.items():
    if not url.strip():
        skipped.append(app)
        continue
    create_connection(app, url.strip())
    created.append(app)

print()
print(f"created ({len(created)}): {created}")
print(f"skipped ({len(skipped)}): {skipped}")

## Verify

In [0]:
display(
    spark.sql("SHOW CONNECTIONS").filter(f"name LIKE '{connection_prefix}%'")
)

In [0]:
for mcp in ENDPOINTS:
    uc_connection = f"climb_hls_{mcp}"
    print(uc_connection)
    spark.sql(f"DROP CONNECTION {uc_connection};")

In [0]:
for mcp in ENDPOINTS:
    uc_connection = f"climb_{mcp}"
    print(uc_connection)
    spark.sql(f"GRANT USE CONNECTION ON CONNECTION {uc_connection} TO `account users`")

In [0]:
%sql
GRANT USE CONNECTION ON CONNECTION climb_preprint TO `account users`;
GRANT USE CONNECTION ON CONNECTION climb_openfda TO `account users`;
GRANT USE CONNECTION ON CONNECTION climb_cms_coverage TO `account users`;
